In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

scope_name = "energy-demand"

existing_scopes = {scope.name for scope in w.secrets.list_scopes()}

if scope_name not in existing_scopes:
    w.secrets.create_scope(scope=scope_name)

print("Secret scope ready:", scope_name)

In [0]:
from getpass import getpass

token = getpass("Enter ENTSO-E token:")

if not token.strip():
    raise ValueError("Token cant be empty")

w.secrets.put_secret(
    scope="energy-demand",
    key="entsoe-token",
    string_value=token.strip(),
)

del token
print("Token saved")

In [0]:
entsoe_token = dbutils.secrets.get(
    scope="energy-demand",
    key="entsoe-token",
)

print("read successfully ", bool(entsoe_token))

In [0]:
from datetime import datetime, timedelta, timezone
import requests

# Request yesterday's data using UTC timestamps
end = datetime.now(timezone.utc).replace(
    hour=0, minute=0, second=0, microsecond=0
)
start = end - timedelta(days=1)

params = {
    "securityToken": entsoe_token,
    "documentType": "A65",       # Total load
    "processType": "A16",        # Actual values
    "outBiddingZone_Domain": "10Y1001A1001A46L",  # SE3
    "periodStart": start.strftime("%Y%m%d%H%M"),
    "periodEnd": end.strftime("%Y%m%d%H%M"),
}

try:
    response = requests.get(
        "https://web-api.tp.entsoe.eu/api",
        params=params,
        timeout=60,
    )
except requests.RequestException:
    raise RuntimeError(
        "Request failed. Check network connectivity or try again later."
    ) from None

print("HTTP status code:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(
        f"ENTSO-E request failed: HTTP {response.status_code}"
    )

raw_xml = response.text
print("Response length (characters):", len(raw_xml))

In [0]:
import xml.etree.ElementTree as ET

root = ET.fromstring(raw_xml)

points = root.findall(".//{*}Point")
reasons = root.findall(".//{*}Reason/{*}text")

print("Document type:", root.tag.split("}")[-1])
print("Number of data points:", len(points))

for reason in reasons:
    print("API message:", reason.text)

if not points:
    raise ValueError("No load data points found.")

In [0]:
from uuid import uuid4
from datetime import datetime, timezone

# uuid is to generate the unique id for data ingestion: ingestion_id
# request_start and request_end are to track the time window of the request

record = [{
    "ingestion_id": str(uuid4()),
    "source": "entsoe",
    "area": "SE3",
    "request_start": start,
    "request_end": end,
    "ingested_at": datetime.now(timezone.utc),
    "http_status": response.status_code,
    "raw_xml": raw_xml,
}]

df = spark.createDataFrame(record)

table_name = "workspace.energy_bronze.entsoe_raw"

df.write.format("delta").mode("append").saveAsTable(table_name)
# write to table in the formate of delta; mode("append") means append the new data to the existing table; saveAsTable(table_name) means save the data to the table")

print("Raw response saved to:", table_name)

In [0]:
%sql
SELECT
    ingestion_id,
    area,
    request_start,
    request_end,
    ingested_at,
    http_status,
    length(raw_xml) AS xml_characters
FROM workspace.energy_bronze.entsoe_raw
ORDER BY ingested_at DESC
-- sorting based on the ingested_at column in descending order
LIMIT 5;

In [0]:
# Read the saved response from Bronze
saved = (
    spark.table("workspace.energy_bronze.entsoe_raw")
    .orderBy("ingested_at", ascending=False)
    .first()
)

root = ET.fromstring(saved["raw_xml"])

for series_number, series in enumerate(
    root.findall(".//{*}TimeSeries"), start=1
):
    print(f"\nTime series {series_number}")
    print("Unit:", series.findtext("{*}quantity_Measure_Unit.name"))
    print("Curve type:", series.findtext("{*}curveType"))

    for period in series.findall("{*}Period"):
        positions = [
            int(point.findtext("{*}position"))
            for point in period.findall("{*}Point")
        ]

        print("Start:", period.findtext("{*}timeInterval/{*}start"))
        print("End:", period.findtext("{*}timeInterval/{*}end"))
        print("Resolution:", period.findtext("{*}resolution"))
        print("Point count:", len(positions))
        print("Positions:", positions)

In [0]:
# Inspect document-level metadata
print("=== Document level ===")
print("Document ID:", root.findtext("{*}mRID"))
print("Revision number:", root.findtext("{*}revisionNumber"))
print("Document type:", root.findtext("{*}type"))
print("Created datetime:", root.findtext("{*}createdDateTime"))

# Inspect each TimeSeries
print("\n=== TimeSeries level ===")

for i, series in enumerate(root.findall(".//{*}TimeSeries"), start=1):
    print(f"\nTimeSeries {i}")
    print("Series ID:", series.findtext("{*}mRID"))
    print("Business type:", series.findtext("{*}businessType"))
    print("Curve type:", series.findtext("{*}curveType"))
    print(
        "Area:",
        series.findtext("{*}outBiddingZone_Domain.mRID")
        or series.findtext("{*}inBiddingZone_Domain.mRID")
    )

In [0]:
%sql
SELECT
    ingestion_id,
    ingested_at,
    request_start,
    request_end
FROM workspace.energy_bronze.entsoe_raw
ORDER BY ingested_at DESC;

In [0]:
import xml.etree.ElementTree as ET

# Inspect a small sample of saved responses
recent = (
    spark.table("workspace.energy_bronze.entsoe_raw")
    .orderBy("ingested_at", ascending=False)
    .limit(5)
    .collect()
)

for row in recent:
    document = ET.fromstring(row["raw_xml"])

    print("\nIngestion ID:", row["ingestion_id"])
    print("Request range:", row["request_start"], "→", row["request_end"])
    print("Ingested at:", row["ingested_at"])
    print("Document ID:", document.findtext("{*}mRID"))
    print("Revision:", document.findtext("{*}revisionNumber"))
    print("Created at:", document.findtext("{*}createdDateTime"))